In [1]:
import os

In [2]:
def list_pdf_files(directory):
    """
    Recursively lists all PDF files in a directory and its subdirectories.
    
    :param directory: The root directory to search.
    :return: A list of paths to PDF files.
    """
    pdf_files = []
    
    for root, _, files in os.walk(directory):
        for file in files:
            if file.lower().endswith(".pdf"):
                pdf_files.append(os.path.join(root, file))
    
    return pdf_files

In [3]:
# Example usage:
# directory_path = "/path/to/directory"
# pdf_list = list_pdf_files(directory_path)
# print("Found PDFs:", pdf_list)

In [4]:
directory_path = "./material/"

In [5]:
pdf_list = list_pdf_files(directory_path)

In [6]:
pdf_list[:2]

['./material/EGO1/0-EGo1资料文档-v1.1/EGo1-电路图/EGO1电路图.pdf',
 './material/EGO1/0-EGo1资料文档-v1.1/EGo1-硬件手册/Ego1_UserManual_v2.2.pdf']

In [7]:
import fitz  # PyMuPDF
import io
from PIL import Image
import pytesseract
import camelot
import logging

In [12]:
import os
os.environ["TESSDATA_PREFIX"] = "/usr/share/tesseract-ocr/4.00/tessdata/"

In [13]:
# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [14]:
def extract_pdf_content(pdf_path):
    """
    Extracts textual content, OCR results from images, and tables from a PDF file.
    
    Steps performed:
      1. Parse PDF and extract raw text and images.
      2. Process each embedded image with OCR (using pytesseract) to extract additional text.
      3. Extract tables using Camelot.
    
    Parameters:
        pdf_path (str): Path to the PDF file.
    
    Returns:
        dict: A dictionary with keys:
            - 'raw_text': Combined text extracted directly from the PDF.
            - 'ocr_text': Combined OCR text extracted from images.
            - 'tables': A list of tables (each represented as a pandas DataFrame) extracted from the PDF.
    """
    
    # Containers for our results
    raw_text_content = []
    ocr_text_content = []
    extracted_tables = []
    
    # Open the PDF file using PyMuPDF
    try:
        doc = fitz.open(pdf_path)
        logger.info(f"Opened PDF: {pdf_path} with {doc.page_count} pages.")
    except Exception as e:
        logger.error(f"Error opening PDF: {e}")
        return None
    
    # Process each page
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        logger.info(f"Processing page {page_num+1}/{doc.page_count}")
        
        # Extract raw text from the page
        page_text = page.get_text("text")
        raw_text_content.append(f"--- Page {page_num+1} ---\n{page_text}")
        
        # Extract images from the page
        image_list = page.get_images(full=True)
        logger.info(f"Found {len(image_list)} images on page {page_num+1}.")
        
        for img_index, img in enumerate(image_list):
            xref = img[0]
            try:
                base_image = doc.extract_image(xref)
                image_bytes = base_image["image"]
                image_ext = base_image["ext"]
                image = Image.open(io.BytesIO(image_bytes))
                
                # Run OCR on the image
                ocr_text = pytesseract.image_to_string(image)
                if ocr_text.strip():
                    ocr_text_content.append(
                        f"--- OCR from Page {page_num+1}, Image {img_index+1} ({image_ext}) ---\n{ocr_text}"
                    )
                else:
                    logger.info(f"No OCR text found in Page {page_num+1}, Image {img_index+1}.")
            except Exception as e:
                logger.error(f"Error processing image on page {page_num+1}, image {img_index+1}: {e}")
    
    # Use Camelot to extract tables from the PDF
    # (Camelot works best with PDFs that are not scanned images.)
    try:
        logger.info("Extracting tables using Camelot...")
        # You can adjust 'pages' parameter as needed; here we use all pages.
        tables = camelot.read_pdf(pdf_path, pages="all", flavor="lattice")  # or flavor="stream"
        logger.info(f"Found {len(tables)} table(s) in the PDF.")
        for table in tables:
            extracted_tables.append(table.df)  # table.df is a pandas DataFrame
    except Exception as e:
        logger.error(f"Error extracting tables with Camelot: {e}")
    
    # Combine the results in a dictionary
    results = {
        "raw_text": "\n".join(raw_text_content),
        "ocr_text": "\n".join(ocr_text_content),
        "tables": extracted_tables,
    }
    
    return results

In [15]:
result = extract_pdf_content(pdf_list[1])

INFO:__main__:Opened PDF: ./material/EGO1/0-EGo1资料文档-v1.1/EGo1-硬件手册/Ego1_UserManual_v2.2.pdf with 21 pages.
INFO:__main__:Processing page 1/21
INFO:__main__:Found 2 images on page 1.
INFO:__main__:Processing page 2/21
INFO:__main__:Found 1 images on page 2.
INFO:__main__:Processing page 3/21
INFO:__main__:Found 2 images on page 3.
INFO:__main__:No OCR text found in Page 3, Image 2.
INFO:__main__:Processing page 4/21
INFO:__main__:Found 2 images on page 4.
INFO:__main__:Processing page 5/21
INFO:__main__:Found 3 images on page 5.
INFO:__main__:No OCR text found in Page 5, Image 2.
INFO:__main__:Processing page 6/21
INFO:__main__:Found 4 images on page 6.
INFO:__main__:No OCR text found in Page 6, Image 2.
INFO:__main__:No OCR text found in Page 6, Image 4.
INFO:__main__:Processing page 7/21
INFO:__main__:Found 2 images on page 7.
INFO:__main__:Processing page 8/21
INFO:__main__:Found 2 images on page 8.
INFO:__main__:Processing page 9/21
INFO:__main__:Found 2 images on page 9.
INFO:__ma

In [22]:
#print(result["raw_text"])

In [23]:
#print(result["ocr_text"])

In [37]:
from pdf2image import convert_from_path
import re

In [42]:
# Define patterns to remove
UNWANTED_PATTERNS = [
    r"依元素科技有限公司",
    r"Xilinx 全球合作伙伴",
    r"www\.e-elements\.com",
    r"EGO1 User Manual"
]

def clean_text(text):
    """Removes unwanted recurring elements from the extracted text."""
    for pattern in UNWANTED_PATTERNS:
        text = re.sub(pattern, "", text)
    return text.strip()  # Remove leading/trailing spaces

In [43]:
def extract_chinese_text_combined(pdf_path):
    """
    Extracts text from a PDF, using PyMuPDF for embedded text and Tesseract OCR for image-based text.

    Parameters:
        pdf_path (str): Path to the PDF file.

    Returns:
        str: Extracted text from the PDF.
    """
    doc = fitz.open(pdf_path)  # Open the PDF
    extracted_text = []
    
    try:
        images = convert_from_path(pdf_path, dpi=300)  # Convert PDF pages to images
    except Exception as e:
        print(f"Error converting PDF to images: {e}")
        images = None  # If image conversion fails, OCR will be skipped.

    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text = page.get_text("text")  # Extract embedded text

        # If no text was extracted, fall back to OCR
        if not text.strip() and images:
            try:
                text = pytesseract.image_to_string(images[page_num], lang="chi_sim")  # Use Chinese OCR
            except Exception as e:
                print(f"OCR failed on page {page_num + 1}: {e}")
                text = ""
        # Clean unwanted recurring elements
        text = clean_text(text)
        extracted_text.append(f"--- Page {page_num+1} ---\n{text}")

    return "\n".join(extracted_text)

In [44]:
# Example Usage:
chinese_text_combined = extract_chinese_text_combined(pdf_list[1])
print(chinese_text_combined[:1000])

--- Page 1 ---
EGO1 用户手册
2018.04
ver2.2
--- Page 2 ---
目录
1.
概述.............................................................................................................................................
2
2.
FPGA............................................................................................................................................
2
3.
板卡供电.....................................................................................................................................
3
4.
系统时钟.....................................................................................................................................
3
5.
FPGA 配置...................................................................................................................................
3
6.
通用I/O 接口..............................................................................................................................
4
6.1 按键...................................................................

In [45]:
print(chinese_text_combined)

--- Page 1 ---
EGO1 用户手册
2018.04
ver2.2
--- Page 2 ---
目录
1.
概述.............................................................................................................................................
2
2.
FPGA............................................................................................................................................
2
3.
板卡供电.....................................................................................................................................
3
4.
系统时钟.....................................................................................................................................
3
5.
FPGA 配置...................................................................................................................................
3
6.
通用I/O 接口..............................................................................................................................
4
6.1 按键...................................................................

In [17]:
if result:
    # Print out a summary of what was extracted.
    print("=== Raw Text ===")
    print(result["raw_text"][:500], "...\n")  # print first 500 chars
    
    print("=== OCR Text ===")
    print(result["ocr_text"][:500], "...\n")
    
    print("=== Tables ===")
    for idx, table_df in enumerate(result["tables"], start=1):
        print(f"Table {idx}:")
        print(table_df.head(), "\n")

=== Raw Text ===
--- Page 1 ---
EGO1 用户手册
2018.04
ver2.2
依元素科技有限公司

--- Page 2 ---
EGO1 User Manual
目录
1.
概述.............................................................................................................................................
2
2.
FPGA............................................................................................................................................
2
3.
板卡供电............................................................................................................ ...

=== OCR Text ===
--- OCR from Page 1, Image 1 (jpeg) ---
Ce = erements

--- OCR from Page 1, Image 2 (jpeg) ---
=. XILINX

UNIVERSITY PROGRAM
PARTNER

--- OCR from Page 2, Image 1 (jpeg) ---
Ce &-eLements

--- OCR from Page 3, Image 1 (jpeg) ---
Ce &-eLements

--- OCR from Page 4, Image 1 (jpeg) ---
Ce &-eLements

--- OCR from Page 4, Image 2 (jpeg) ---
Part Number XC7AI2T ——XC7A1ST_——xcTA2ST —-['XCTASST

Uupecete 1200 6640 2360 | 33,200
hee slices 2,000 2,600 3,650 5,200


In [ ]:




# Example usage:
if __name__ == "__main__":
    pdf_file = "path/to/your/manual.pdf"
    
    
    if result:
        # Print out a summary of what was extracted.
        print("=== Raw Text ===")
        print(result["raw_text"][:500], "...\n")  # print first 500 chars
        
        print("=== OCR Text ===")
        print(result["ocr_text"][:500], "...\n")
        
        print("=== Tables ===")
        for idx, table_df in enumerate(result["tables"], start=1):
            print(f"Table {idx}:")
            print(table_df.head(), "\n")
